In [ ]:
import numpy as np
import pandas as pd
import random

In [ ]:
class MyLogReg():
    def __init__(self, n_iter=10, learning_rate=0.1, weights=None, metric = None, reg = None, l1_coef=0, l2_coef=0, sgd_sample = None, random_state = 42):
        self.n_iter = n_iter
        self.learning_rate = learning_rate
        self.weights = weights
        self.eps = 1e-15
        self._sigmoid = lambda z: 1 / (1 + np.exp(-np.clip(z, -500, 500)))
        self.metric = metric
        self.random_state = random_state
        self.sgd_sample = sgd_sample

        if reg == None:
            self.l1_coef = 0
            self.l2_coef = 0
        elif reg == 'l1':
            self.l2_coef = 0
            self.l1_coef = l1_coef
        elif reg == 'l2':
            self.l1_coef = 0
            self.l2_coef = l2_coef
        elif reg == 'elasticnet':
            self.l1_coef = l1_coef
            self.l2_coef = l2_coef
        
        

    def fit(self, X, y, verbose=False):
        X_with_bias = X.copy()
        X_with_bias.insert(0, 'x0', 1)
        X_matrix = X_with_bias.values
        n_samples, count_of_features = X_matrix.shape
        random.seed(self.random_state)
            
        
        if self.weights is None:
            self.weights = np.ones(count_of_features)
        
        for i in range(1, self.n_iter + 1):

            if self.sgd_sample is not None:
                if self.sgd_sample >= 1:
                    sample_rows_idx = random.sample(range(X_matrix.shape[0]), self.sgd_sample)
                else:
                    sample_rows_idx = random.sample(range(X_matrix.shape[0]), int(self.sgd_sample * X_matrix.shape[0]))
            else:
                sample_rows_idx = range(X_matrix.shape[0])

            X_batch = X_matrix[sample_rows_idx]
            y_batch = y.values[sample_rows_idx]

            if callable(self.learning_rate):
                curr_lr = self.learning_rate(i)
            else:
                curr_lr = self.learning_rate

            logits = X_batch.dot(self.weights)
            probs = self._sigmoid(logits)
            regularisation = self.l1_coef * np.sum(np.abs(self.weights[1:])) + self.l2_coef * np.sum(self.weights[1:] ** 2)
            #loss
            LogLoss = -np.mean(y_batch.dot(np.log(self.eps + probs))
                                + (1 - y_batch).dot(np.log(self.eps + 1 - probs))) + regularisation
            #w - grad
            self.weights = self.weights - curr_lr * (((probs - y_batch).dot(X_batch))/len(sample_rows_idx) \
                + self.l1_coef * np.sign(self.weights) + self.l2_coef * 2 * self.weights)


        if self.metric is not None:  
            y_pred = self.predict(X) 
            metric_func = getattr(self, self.metric)
    
            if self.metric in ['accuracy', 'precision', 'recall', 'f1']:
                y_pred = self.predict(X)
                self.best_metric = metric_func(y_pred, y)
            elif self.metric == 'roc_auc':
                y_proba = self.predict_proba(X)
                self.best_metric = metric_func(y_proba, y)
    
            if verbose and i % verbose == 0:
                print(f"step {i} | loss = {LogLoss} | metric = {self.best_metric}")
            else:
                if verbose and i % verbose == 0:
                    print(f"step {i} | loss = {LogLoss} | lr = {curr_lr}")

    def predict_proba(self, X):
        if callable(X):
            X = X()
    
        if X.shape[1] == len(self.weights):  
            X_with_bias = X
        else:
            if isinstance(X, np.ndarray):
                X = pd.DataFrame(X)
            X_with_bias = X.copy()
            X_with_bias.insert(0, 'x0', 1)
            X_with_bias = X_with_bias.values
    
        logits = X_with_bias.dot(self.weights)
        return self._sigmoid(logits)
    
    def predict(self, X):
        predict = self.predict_proba(X)
        for i in range(len(predict)):
            if predict[i] > 0.5:
                predict[i] = 1
            else:
                predict[i] = 0
        return predict.astype('int64')
    
    def get_coef(self):
        return self.weights[1:]
    
    def accuracy(self, y_pred, y_true):
        return 1 - np.mean(y_pred == y_true)

    def precision(self, y_pred, y_true):
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        return tp / (tp + fp) if (tp + fp) > 0 else 0
                
    def recall(self, y_pred, y_true):
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fn = np.sum((y_pred == 0) & (y_true == 1))
        return tp / (tp + fn) if (tp + fn) > 0 else 0

    def f1(self, X_matrix, y_true, beta = 1):
        prec = self.precision(X_matrix, y_true)
        rec = self.recall(X_matrix, y_true)
        return (1 + beta ** 2) * prec * rec / (prec + rec)

    def roc_auc(self, y_proba, y_true):
        idx = y_proba.argsort()[::-1]
        y_true_sorted = y_true[idx]
    
        pos = np.sum(y_true == 1)
        neg = np.sum(y_true == 0)
    
        pos_ranks = np.where(y_true_sorted == 1)[0] + 1
        return 1 - (np.sum(pos_ranks) - pos*(pos+1)/2) / (pos * neg)
        
    def get_best_score(self):
        return self.best_metric
            
    def __str__(self):
        return f"MyLogReg class: n_iter={self.n_iter}, learning_rate={self.learning_rate}"
